# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [28]:
import os

In [29]:
# Write your code below.
from dotenv import load_dotenv
load_dotenv()

True

In [30]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [31]:
import os
from glob import glob

# Write your code below.
price_data_dir = os.environ["PRICE_DATA"]
all_files = glob(os.path.join(price_data_dir, "*.csv"))
all_files

[]

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [32]:
# Write your code below.
# If all_files is empty, you may need to update it to include parquet files, e.g.:
all_files = glob(os.path.join(price_data_dir, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(all_files).set_index("ticker")
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Adj_Close_lag_1 = x['Adj Close'].shift(1))
)
dd_feat = dd_feat.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)
dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(hi_lo_range = lambda x: x['High'] - x['Low'])
)
dd_feat

C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:8: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:14: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is une

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
npartitions=90,,,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [33]:
# Write your code below.

pd_feat = dd_feat.compute()
pd_feat['ma_returns_10'] = pd_feat.groupby('ticker')['Returns'].rolling(10).mean().reset_index(level=0, drop=True)

pd_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ma_returns_10
ticker,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN


In [35]:
pd_feat.sample(10)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ma_returns_10
ticker,,,,,,,,,,,,,,
PBI,1996-05-02,24.125000,24.437500,24.000000,24.375000,8.192500,390800.0,PBI.csv,1996,24.375000,8.192500,0.000000,0.437500,0.001574
SPDN,2016-11-22,37.820000,37.948002,37.779999,37.840000,36.538239,56400.0,SPDN.csv,2016,37.919998,36.615486,-0.002110,0.168003,-0.003007
BGS,2009-04-15,5.040000,5.210000,5.040000,5.180000,2.828593,58500.0,BGS.csv,2009,5.080000,2.773987,0.019685,0.170000,-0.000209
GAZ,2018-10-29,43.209999,43.209999,43.209999,43.209999,43.209999,0.0,GAZ.csv,2018,43.209999,43.209999,0.000000,0.000000,-0.001089
SMG,2018-01-31,92.470001,92.800003,87.000000,90.269997,85.353813,2710900.0,SMG.csv,2018,91.959999,86.951782,-0.018378,5.800003,-0.015364
SPXC,2017-06-20,24.940001,25.230000,24.590000,25.020000,25.020000,363500.0,SPXC.csv,2017,25.190001,25.190001,-0.006749,0.639999,0.001470
GAZ,2016-07-29,0.570000,0.600000,0.550000,0.560000,0.560000,58100.0,GAZ.csv,2016,0.550000,0.550000,0.018182,0.050000,0.014802
ATLC,2015-02-23,2.790000,2.900000,2.790000,2.900000,2.900000,700.0,ATLC.csv,2015,2.880000,2.880000,0.006944,0.110000,0.000028
MOH,2010-07-14,18.986666,20.193333,18.986666,20.139999,20.139999,899700.0,MOH.csv,2010,18.906666,18.906666,0.065233,1.206667,0.002787


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

 It is not necessary to convert to pandas to calculate the moving average return. Dask can perform the same operations. If the dataset is large then using Dask is better for such operations because it runs them in parallel. The rolling and groupby operations on Dask are slower than pandas so it is better to use pandas for small datasets.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.